In [17]:
import os

import pandas as pd
import geopandas as gpd
from tqdm import tqdm

In [18]:
# CELL 2 — Load the Geo Starting Row Index and Filter Toronto ADAs

geo_index = pd.read_csv("../../data/census/ada/98-401-X2021012_Geo_starting_row.CSV",
                        dtype=str)

geo_index["Line Number"] = geo_index["Line Number"].astype(int)

# Region prefixes we want
prefixes = [
    # "3525",  # Hamilton
    # "3524",  # Halton
    # "3521",  # Peel
    # "3519",  # York
    "3520",  # Toronto
    # "3518",  # Durham
]

mask = geo_index["Geo Name"].str.startswith(tuple(prefixes))
toronto_geo = geo_index[mask].copy()

print(f"Toronto ADA count: {len(toronto_geo)}")

ontario_geo = toronto_geo


Toronto ADA count: 279


In [19]:
# CELL 3 — Sort ADAs by row order and compute row ranges

ontario_geo = ontario_geo.sort_values("Line Number").reset_index(drop=True)

# Compute row ranges (start row, end row)
start_rows = ontario_geo["Line Number"].astype(int).tolist()
end_rows = start_rows[1:] + [None]

row_ranges = list(zip(start_rows, end_rows))

row_ranges[:5]
# NOTE: the last range intentionally runs to EOF (nrows=None handles this below)


[(5264633, 5267264),
 (5267264, 5269895),
 (5269895, 5272526),
 (5272526, 5275157),
 (5275157, 5277788)]

In [ ]:
# CELL 3.5 — Define characteristic codes and ID ranges

# Mapping from CHARACTERISTIC_ID to semantic short codes.
# Filtering to a subset is now done in prepare_ada_data_all.ipynb (ACTIVE_IDS constant).
CHARACTERISTIC_CODES = {
    # Demographics - Population
    1: 'pop_2021',  # Population, 2021
    2: 'pop_2016',  # Population, 2016
    3: 'pop_pct_change',  # Population % change, 2016 to 2021
    6: 'pop_density',  # Population density per square kilometre

    # Age structure
    34: 'age_total_dist',  # Total - Distribution (%) of the population by broad age groups
    35: 'age_0_14',  # 0 to 14 years
    36: 'age_15_64',  # 15 to 64 years
    37: 'age_65_over',  # 65 years and over
    38: 'age_85_over',  # 85 years and over
    39: 'age_mean',  # Average age of the population
    40: 'age_median',  # Median age of the population

    # Household composition
    50: 'pvt_house_total',  # Total - Private households by household size
    51: 'pvt_house_1',  # 1 person
    52: 'pvt_house_2',  # 2 persons
    53: 'pvt_house_3',  # 3 persons
    54: 'pvt_house_4',  # 4 persons
    55: 'pvt_house_5plus',  # 5 or more persons
    56: 'pvt_house_persons',  # Number of persons in private households
    57: 'pvt_house_avg_size',  # Average household size

    # Income - Core measures
    111: 'income_total',  # Total - Income statistics (population aged 15+)
    112: 'income_total_recip',  # Number of total income recipients
    113: 'income_total_median',  # Median total income among recipients ($)
    114: 'income_aftertax_recip',  # Number of after-tax income recipients
    115: 'income_aftertax_median',  # Median after-tax income among recipients ($)
    116: 'income_market_recip',  # Number of market income recipients
    117: 'income_market_median',  # Median market income among recipients ($)
    118: 'income_emp_recip',  # Number of employment income recipients
    119: 'income_emp_median',  # Median employment income among recipients ($)
    120: 'income_govtrans_recip',  # Number of government transfers recipients
    121: 'income_govtrans_median',  # Median government transfers among recipients ($)
    122: 'income_ei_recip',  # Number of employment insurance benefits recipients
    123: 'income_ei_median',  # Median employment insurance benefits among recipients ($)
    124: 'income_covid_recip',  # Number of COVID-19 emergency and recovery benefits recipients
    125: 'income_covid_median',  # Median COVID-19 emergency and recovery benefits ($)

    # Income - Poverty and distribution
    345: 'lim_at_prev',  # Prevalence of low income based on LIM-AT (%)
    346: 'lim_at_0_17',  # LIM-AT prevalence, age 0 to 17 (%)
    347: 'lim_at_0_5',  # LIM-AT prevalence, age 0 to 5 (%)
    348: 'lim_at_18_64',  # LIM-AT prevalence, age 18 to 64 (%)
    349: 'lim_at_65_over',  # LIM-AT prevalence, age 65 years and over (%)
    365: 'income_decile_total',  # Total - Adjusted after-tax economic family income decile
    366: 'income_decile_bottom_half',  # In bottom half of the distribution
    367: 'income_decile_1',  # In bottom decile
    368: 'income_decile_2',  # In second decile
    369: 'income_decile_3',  # In third decile
    370: 'income_decile_4',  # In fourth decile
    371: 'income_decile_5',  # In fifth decile
    372: 'income_decile_top_half',  # In top half of the distribution
    373: 'income_decile_6',  # In sixth decile
    374: 'income_decile_7',  # In seventh decile
    375: 'income_decile_8',  # In eighth decile
    376: 'income_decile_9',  # In ninth decile
    377: 'income_decile_10',  # In top decile
    378: 'gini_measures',  # Total - Inequality measures (population count)
    379: 'gini_total_income',  # Gini index on adjusted household total income
    380: 'gini_market_income',  # Gini index on adjusted household market income
    381: 'gini_aftertax_income',  # Gini index on adjusted household after-tax income
    382: 'gini_p90_p10',  # P90/P10 ratio on adjusted household after-tax income

    # Housing - Tenure and affordability
    1414: 'housing_tenure_total',  # Total - Private households by tenure
    1415: 'housing_tenure_owner',  # Owner
    1416: 'housing_tenure_renter',  # Renter
    1417: 'housing_tenure_provided',  # Dwelling provided by local government, First Nation or Indian band
    1418: 'housing_condo_total',  # Total - Occupied private dwellings by condominium status
    1465: 'housing_shelter_total',  # Total - Owner and tenant households by shelter-cost-to-income ratio
    1466: 'housing_shelter_under30',  # Spending less than 30% of income on shelter costs
    1467: 'housing_shelter_30plus',  # Spending 30% or more of income on shelter costs
    1468: 'housing_shelter_30_100',  # Spending 30% to less than 100% of income on shelter costs

    # Housing - Core need
    1479: 'housing_core_need_total',  # Total - Owner and tenant households (core need denominator)
    1480: 'housing_core_need_yes',  # In core housing need
    1481: 'housing_core_need_no',  # Not in core housing need
    1482: 'housing_owner_total',  # Total - Owner households in non-farm, non-reserve private dwellings
    1483: 'housing_owner_mortgage_pct',  # % of owner households with a mortgage
    1484: 'housing_owner_shelter_30plus_pct',  # % of owner households spending 30%+ on shelter costs
    1485: 'housing_owner_core_need_pct',  # % of owners in core housing need

    # Citizenship
    1522: 'citizen_total',  # Total - Citizenship for the population in private households
    1523: 'citizen_canadian',  # Canadian citizens
    1524: 'citizen_canadian_under18',  # Canadian citizens aged under 18
    1525: 'citizen_canadian_18over',  # Canadian citizens aged 18 and over
    1526: 'citizen_not_canadian',  # Not Canadian citizens

    # Visible minorities
    1683: 'visible_minority_total',  # Total - Visible minority for the population in private households
    1684: 'visible_minority_yes',  # Total visible minority population
    1685: 'visible_minority_south_asian',  # South Asian
    1686: 'visible_minority_chinese',  # Chinese
    1687: 'visible_minority_black',  # Black
    1688: 'visible_minority_filipino',  # Filipino
    1689: 'visible_minority_arab',  # Arab
    1690: 'visible_minority_latin_american',  # Latin American
    1691: 'visible_minority_southeast_asian',  # Southeast Asian
    1692: 'visible_minority_west_asian',  # West Asian
    1693: 'visible_minority_korean',  # Korean
    1694: 'visible_minority_japanese',  # Japanese
    1695: 'visible_minority_nie',  # Visible minority, n.i.e.
    1696: 'visible_minority_multiple',  # Multiple visible minorities
    1697: 'visible_minority_no',  # Not a visible minority

    # Education - all ages 15+
    1998: 'education_total',  # Total - Highest certificate, diploma or degree (age 15+)
    1999: 'education_none',  # No certificate, diploma or degree
    2000: 'education_secondary',  # High (secondary) school diploma or equivalency certificate
    2001: 'education_postsec',  # Postsecondary certificate, diploma or degree
    2002: 'education_postsec_below_bachelor',  # Postsecondary certificate or diploma below bachelor level
    2003: 'education_apprentice_trades',  # Apprenticeship or trades certificate or diploma
    2004: 'education_apprentice_non',  # Non-apprenticeship trades certificate or diploma
    2005: 'education_apprentice_cert',  # Apprenticeship certificate
    2006: 'education_college',  # College, CEGEP or other non-university certificate or diploma
    2007: 'education_university_below_bachelor',  # University certificate or diploma below bachelor level
    2008: 'education_bachelor_higher',  # Bachelor's degree or higher
    2009: 'education_bachelor',  # Bachelor's degree
    2010: 'education_university_above_bachelor',  # University certificate or diploma above bachelor level
    2011: 'education_medical_dental_vet',  # Degree in medicine, dentistry, veterinary medicine or optometry
    2012: 'education_masters',  # Master's degree
    2013: 'education_doctorate',  # Earned doctorate degree

    # Education - ages 25 to 64
    2014: 'education_25_64_total',  # Total - Highest certificate, diploma or degree (age 25-64)
    2015: 'education_25_64_none',  # No certificate, diploma or degree (age 25-64)
    2016: 'education_25_64_secondary',  # High school diploma or equivalency (age 25-64)
    2017: 'education_25_64_postsec',  # Postsecondary certificate, diploma or degree (age 25-64)
    2018: 'education_25_64_postsec_below_bachelor',  # Postsecondary below bachelor level (age 25-64)
    2019: 'education_25_64_apprentice_trades',  # Apprenticeship or trades certificate (age 25-64)
    2020: 'education_25_64_apprentice_non',  # Non-apprenticeship trades certificate (age 25-64)
    2021: 'education_25_64_apprentice_cert',  # Apprenticeship certificate (age 25-64)
    2022: 'education_25_64_college',  # College, CEGEP or non-university certificate (age 25-64)
    2023: 'education_25_64_university_below_bachelor',  # University certificate below bachelor (age 25-64)
    2024: 'education_25_64_bachelor_higher',  # Bachelor's degree or higher (age 25-64)
    2025: 'education_25_64_bachelor',  # Bachelor's degree (age 25-64)
    2026: 'education_25_64_university_above_bachelor',  # University certificate above bachelor (age 25-64)
    2027: 'education_25_64_medical_dental_vet',  # Degree in medicine, dentistry or veterinary (age 25-64)
    2028: 'education_25_64_masters',  # Master's degree (age 25-64)
    2029: 'education_25_64_doctorate',  # Earned doctorate degree (age 25-64)

    # Labour - Occupation
    2246: 'labour_occupation_total',  # Total - Labour force by occupation
    2247: 'labour_occupation_na',  # Occupation - not applicable
    2248: 'labour_occupation_all',  # All occupations
    2249: 'labour_occupation_0_management',  # 0 Legislative and senior management
    2250: 'labour_occupation_1_business_finance',  # 1 Business, finance and administration
    2251: 'labour_occupation_2_science',  # 2 Natural and applied sciences and related
    2252: 'labour_occupation_3_health',  # 3 Health occupations
    2253: 'labour_occupation_4_education_law_social',  # 4 Education, law and social, community services
    2254: 'labour_occupation_5_arts_culture',  # 5 Arts, culture, recreation and sport
    2255: 'labour_occupation_6_sales_service',  # 6 Sales and service occupations
    2256: 'labour_occupation_7_trades_transport',  # 7 Trades, transport and equipment operators
    2257: 'labour_occupation_8_natural_resources',  # 8 Natural resources, agriculture and related
    2258: 'labour_occupation_9_manufacturing',  # 9 Occupations in manufacturing and utilities

    # Labour - Industry
    2259: 'labour_industry_total',  # Total - Labour force by industry sector
    2260: 'labour_industry_na',  # Industry - not applicable
    2261: 'labour_industry_all',  # All industries
    2262: 'labour_industry_11_agriculture',  # 11 Agriculture, forestry, fishing and hunting
    2263: 'labour_industry_21_mining',  # 21 Mining, quarrying, and oil and gas extraction
    2264: 'labour_industry_22_utilities',  # 22 Utilities
    2265: 'labour_industry_23_construction',  # 23 Construction
    2266: 'labour_industry_31_33_manufacturing',  # 31-33 Manufacturing
    2267: 'labour_industry_41_wholesale',  # 41 Wholesale trade
    2268: 'labour_industry_44_45_retail',  # 44-45 Retail trade
    2269: 'labour_industry_48_49_transportation',  # 48-49 Transportation and warehousing
    2270: 'labour_industry_51_information',  # 51 Information and cultural industries
    2271: 'labour_industry_52_finance',  # 52 Finance and insurance
    2272: 'labour_industry_53_real_estate',  # 53 Real estate and rental and leasing
    2273: 'labour_industry_54_professional',  # 54 Professional, scientific and technical services
    2274: 'labour_industry_55_management',  # 55 Management of companies and enterprises
    2275: 'labour_industry_56_administrative',  # 56 Administrative and support, waste management
    2276: 'labour_industry_61_education',  # 61 Educational services
    2277: 'labour_industry_62_healthcare',  # 62 Health care and social assistance
    2278: 'labour_industry_71_arts',  # 71 Arts, entertainment and recreation
    2279: 'labour_industry_72_accommodation',  # 72 Accommodation and food services
    2280: 'labour_industry_81_other_services',  # 81 Other services (except public administration)
    2281: 'labour_industry_91_public_admin',  # 91 Public administration
}

# Define CHARACTERISTIC_ID ranges (inclusive) by category
ID_RANGES = [
    # Demographic basics - Population counts
    (1, 2),        # Population, 2021 and 2016
    (3, 3),        # Population percentage change, 2016 to 2021
    (6, 6),        # Population density per square kilometre

    # Age structure
    (34, 37),      # Age group distribution (0–14, 15–64, 65+)
    (38, 40),      # Age statistics (85+, mean age, median age)

    # Household composition
    (50, 55),      # Household size distribution (1, 2, 3, 4, 5+)
    (56, 57),      # Persons in households and average size

    # Income - Core measures
    (111, 119),    # Total, after-tax, market, and employment income
    (120, 121),    # Government transfers
    (122, 125),    # Employment insurance and COVID benefits

    # Income - Poverty and distribution
    (345, 349),    # Low Income Measure, after tax (LIM-AT)
    (365, 382),    # Income deciles and Gini coefficient

    # Housing - Tenure and affordability
    (1414, 1418),  # Housing tenure (owner vs renter)
    (1465, 1468),  # Shelter costs >30% of income

    # Housing - Core need
    (1479, 1485),  # Core housing need

    # Education
    (1998, 2029),  # Highest certificate, diploma, or degree

    # Citizenship
    (1522, 1526),  # Citizenship status

    # Visible minorities
    (1683, 1697),  # Visible minority status

    # Labour - Occupation
    (2246, 2258),  # Labour force by occupation

    # Labour - Industry
    (2259, 2281),  # Labour force by industry sector
]


In [21]:
# CELL 4 — Extract each ADA block individually, filter IDs & columns (optimized)

main_file = "../../data/census/ada/98-401-X2021012_English_CSV_data.csv"
output_dir = "../../data/census/ada/toronto"
os.makedirs(output_dir, exist_ok=True)

print("Extracting Toronto ADAs with filtering (read once, save by ADA)...")

# Flatten ranges into a set of valid IDs
valid_ids = set()
for start, end in ID_RANGES:
    valid_ids.update(range(start, end + 1))

required_cols = [
    "GEO_NAME",
    "CHARACTERISTIC_ID",
    "CHARACTERISTIC_NAME",
    "C1_COUNT_TOTAL",
    "C10_RATE_TOTAL",
]

# Get Toronto ADA prefixes
toronto_prefixes = ("3520",)

# Read CSV once, filter for valid IDs and Toronto ADAs
print(f"Reading census data file...")
df = pd.read_csv(
    main_file,
    low_memory=False,
    encoding="latin",
    usecols=required_cols,
    dtype={"GEO_NAME": str}  # Ensure GEO_NAME is read as string
)

# Convert CHARACTERISTIC_ID to numeric for filtering
df["CHARACTERISTIC_ID"] = pd.to_numeric(df["CHARACTERISTIC_ID"], errors="coerce")

# Filter for valid IDs and Toronto ADAs
df_filtered = df[
    (df["CHARACTERISTIC_ID"].isin(valid_ids)) &
    (df["GEO_NAME"].str.startswith(toronto_prefixes))
].copy()

# Keep only required columns
df_filtered = df_filtered[required_cols]

print(f"Extracted {len(df_filtered)} rows across {df_filtered['GEO_NAME'].nunique()} Toronto ADAs")

# Add characteristic code column
df_filtered['CHARACTERISTIC_CODE'] = df_filtered['CHARACTERISTIC_ID'].map(CHARACTERISTIC_CODES)

# Reorder columns: GEO_NAME, CHARACTERISTIC_ID, CHARACTERISTIC_NAME, CHARACTERISTIC_CODE, C1_COUNT_TOTAL, C10_RATE_TOTAL
output_cols = ['GEO_NAME', 'CHARACTERISTIC_ID', 'CHARACTERISTIC_NAME', 'CHARACTERISTIC_CODE', 'C1_COUNT_TOTAL', 'C10_RATE_TOTAL']
df_filtered = df_filtered[output_cols]

# Save each ADA to its own file
print(f"Saving individual ADA files...")
for ada_id in tqdm(df_filtered["GEO_NAME"].unique()):
    output_file = os.path.join(output_dir, f"{ada_id}.csv")
    df_ada = df_filtered[df_filtered["GEO_NAME"] == ada_id][output_cols]
    df_ada.to_csv(output_file, index=False)

print(f"Extraction complete!")

Extracting Toronto ADAs with filtering (read once, save by ADA)...
Reading census data file...
Extracted 44919 rows across 279 Toronto ADAs
Saving individual ADA files...


100%|██████████| 279/279 [00:01<00:00, 217.65it/s]

Extraction complete!


In [22]:
# CELL 5 — Merge all Toronto ADA CSVs into one final CSV

output_dir = "../../data/census/ada/toronto"
final_output = "../../data/census/ada/toronto-ada.csv"

csv_files = sorted(os.listdir(output_dir))
write_header = True  # write header only for first file

with open(final_output, "w", encoding="latin") as outfile:
    for fname in tqdm(csv_files):
        fpath = os.path.join(output_dir, fname)
        with open(fpath, "r", encoding="latin") as infile:
            if write_header:
                outfile.write(infile.read())
                write_header = False
            else:
                next(infile)  # skip header
                outfile.write(infile.read())

print(f"Final merged CSV saved: {final_output}")


100%|██████████| 279/279 [00:00<00:00, 20318.28it/s]

Final merged CSV saved: ../../data/census/ada/toronto-ada.csv


In [23]:
# CELL 6 — Load ADA shapefile, filter for Toronto, save as GeoPackage

shapefile_path = "../../data/geo/lada000b21a_e"
output_gpkg = "../../data/geo/toronto-ada.gpkg"

# Load full Canada ADA shapefile
gdf = gpd.read_file(shapefile_path)

# Filter for Toronto ADAs using ADAUID prefix (same prefixes as Cell 2)
toronto_prefixes = [
    # "3525",  # Hamilton
    # "3524",  # Halton
    # "3521",  # Peel
    # "3519",  # York
    "3520",  # Toronto
    # "3518",  # Durham
]
gdf_toronto = gdf[gdf["ADAUID"].str.startswith(tuple(toronto_prefixes))].copy()

gdf_toronto.to_file(output_gpkg, driver="GPKG")

print(f"Saved Toronto ADA geometries to: {output_gpkg}")
print(f"Number of Toronto ADAs: {len(gdf_toronto)}")


Saved Toronto ADA geometries to: ../../data/geo/toronto-ada.gpkg
Number of Toronto ADAs: 279
